In [37]:
import pandas as pd

movies_df = pd.read_csv("movies_final.csv")
movies_df.head()

,title,genres_list,overview,keywords,Cast_list,Director,vote_average,release_date,original_language,movieId,combined_features
0,Inception,Action Science Fiction Adventure,"Cobb, a skilled thief who commits corporate es...",rescue mission dream airplane paris france vir...,Tim Kelleher Silvie Laguna Natasha Beaumont Kr...,Christopher Nolan,8.364,2010-07-15,en,79132,79132 Inception Action Science Fiction Adventu...
1,Interstellar,Adventure Drama Science Fiction,The adventures of a group of explorers who mak...,rescue future spacecraft race against time art...,Jeff Hephner William Devane Elyes Gabel Topher...,Christopher Nolan,8.417,2014-11-05,en,109487,109487 Interstellar Adventure Drama Science Fi...
2,The Dark Knight,Drama Action Crime Thriller,Batman raises the stakes in his war on crime. ...,joker sadism chaos secret identity crime fight...,Tommy Lister Jr. Edison Chen Beatrice Rosen To...,Christopher Nolan,8.512,2008-07-16,en,58559,58559 The Dark Knight Drama Action Crime Thril...
3,Avatar,Action Adventure Fantasy Science Fiction,"In the 22nd century, a paraplegic Marine is di...",future society culture clash space travel spac...,Carvon Futrell Joel David Moore Jon Curry Laz ...,James Cameron,7.573,2009-12-15,en,72998,72998 Avatar Action Adventure Fantasy Science ...
4,The Avengers,Science Fiction Action Adventure,When an unexpected enemy emerges and threatens...,new york city superhero shield based on comic ...,Haneyuri Nako Mizusawa Marin Rikako Sakata Ai ...,Joss Whedon,7.710,2012-04-25,en,89745,89745 The Avengers Science Fiction Action Adve...


In [92]:
import nltk
import string

from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')




[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LILIP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LILIP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LILIP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
import json
import random
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

with open("intents.json", "r") as file:
    intents_data = json.load(file)

movies_df = pd.read_csv("movies_final.csv")

movies_df["combined_features"] = (
    movies_df["combined_features"]
    .fillna("")
)

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies_df["combined_features"])

In [95]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies(query):

    query_vector = tfidf.transform([query])

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten() # Flatten to 1D array

    top_indices = similarities.argsort()[-5:][::-1] # Get indices of top 5 similar movies

    recommendations = (
        movies_df.iloc[top_indices]["title"]
        .tolist()
    )

    response = f"Based on your interest in '{query.lower()}', I recommend these movies:\n\n"

    for movie in recommendations:
        response += f"- {movie}\n"

    return response

In [89]:
waiting_for_recommendation = False

In [ ]:
# Detectar intención
def get_intent(user_input):

    user_words = set(user_input.lower().split())

    best_match = None
    max_matches = 0

    for intent in intents_data["intents"]:

        for pattern in intent["patterns"]:

            pattern_words = set(pattern.lower().split())

            matches = len(
                user_words.intersection(pattern_words)
            )

            if matches > max_matches:
                max_matches = matches
                best_match = intent

    return best_match

In [90]:
# Procesar mensaje
def process_input(text):

    global waiting_for_recommendation

    if waiting_for_recommendation:

        waiting_for_recommendation = False

        return recommend_movies(text)

    intent = get_intent(text)

    if intent is None:

        noanswer = next(
            item
            for item in intents_data["intents"]
            if item["tag"] == "noanswer"
        )

        return random.choice(
            noanswer["responses"]
        )

    if intent["tag"] == "movies":

        waiting_for_recommendation = True

        return random.choice(
            intent["responses"]
        )

    return random.choice(
        intent["responses"]
    )

In [98]:
# Chatbot
print("🎬 MovieBot is running!")
print("Type 'quit' to exit.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() in ["quit", "exit"]:

        print("Bot: Goodbye!")
        break

    response = process_input(user_input)

    print(f"\nBot: {response}\n")

🎬 MovieBot is running!
Type 'quit' to exit.


Bot: Hi there! What can I do for you?


Bot: Tell me a genre, actor, director, or keyword, and I'll find a movie for you!


Bot: Based on your interest in 'nolan', I recommend these movies:

- The Newcomers
- Facing Nolan
- Boulevard
- It Cuts Deep
- Shredderman Rules


Bot: Goodbye!
